In [14]:
import os 
import requests
from dotenv import load_dotenv
import pandas as pd

load_dotenv()

API_KEY = os.getenv("ALPHA_VANTAGE_API_KEY") #get API key from env file

BASE_URL = "https://www.alphavantage.co/query" #base url for api

def fetch_stock_data(symbol):
    
    """
    Fetch daily stock data from Alpha API
    """

    
    params = {
        "function" : "TIME_SERIES_DAILY",
        "symbol": symbol,
        "apikey" : API_KEY 
    }

    response = requests.get(BASE_URL, params=params)
    
    if response.status_code != 200 :
        raise Exception("API request dailed")

   
    data = response.json() 
    
    if "Error Message" in data:
        raise Exception("Invalid API call")

    if "Note" in data:
        raise Exception("API rate limit exceeded")

    return data 

def parse_stock_data(data) :
    """
    Extract time serise data from API response
    """

    if "Time Series (Daily)" not in data:
        raise Exception("Time Series data not found in API response")
    
    return data["Time Series (Daily)"]

import pandas as pd

def convert_to_df(df) :
    return pd.DataFrame(df).T


def time_serise_analysis_dataset (df) :
    df = df.copy() #copy this dataset so that for delink the referance 
    df = df.drop_duplicates() #drop the duplicate value
    df = df.dropna()  #drop the missing value KAN-39
    df = df.rename(columns={                            
            "1. open" :"open",
            "2. high" :"high" ,  
            "3. low"  :"low" , 
            "4. close" :"close"  ,
            "5. volume" : "volume" }) #rename the column
    df = df.astype({"open":float,"high":float,"low":float,"close":float,"volume":int}).round(2) #reasign the data type and round it by 2

    df.index = pd.to_datetime(df.index) #set index value as date time
    
    df = df.sort_index(ascending=False) # sorted the index

    df.reset_index(inplace=True)
    df.rename(columns={"index":"Date"},inplace=True) #change the index name as date 
    df = df.set_index("Date") #set this cloumn as date time 
    return df
def time_serise_feature_engineering(df) :
    
    df["close_norm"] = (df["close"] /  df["close"].max()).astype(float).round(2)  #normalized this stock value KAN-40 


    df["retuens"] = (df["close"].pct_change().fillna(0)).astype(float).round(4) #calculatethe daily return KAN-41

        

    df["pct_change"] = (df["close"].pct_change().fillna(0) * 100).astype(float).round(2) #calculate the percentage chnage
    
    return df

In [29]:
def compare_stock(symbols: list): 

    dataframes = {}
    for symbol in symbols:
        data = fetch_stock_data(symbol)
        df = parse_stock_data(data)
        df = convert_to_df(df)
        df = time_serise_analysis_dataset(df)
        df = time_serise_feature_engineering(df) 

        dataframes[symbol] = df
    return dataframes

In [30]:
compare_stock(["ACPL","NASDAQ"])

Exception: Invalid API call